In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_parquet('../data/instacart_cleaned_v1.parquet')

In [3]:
df.head(20)

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
0,2,33120,1,1,202279,3,5,9,8.0,Organic Egg Whites,eggs,dairy eggs
1,2,28985,2,1,202279,3,5,9,8.0,Michigan Organic Kale,fresh vegetables,produce
2,2,9327,3,0,202279,3,5,9,8.0,Garlic Powder,spices seasonings,pantry
3,2,45918,4,1,202279,3,5,9,8.0,Coconut Butter,oils vinegars,pantry
4,2,30035,5,0,202279,3,5,9,8.0,Natural Sweetener,baking ingredients,pantry
5,2,17794,6,1,202279,3,5,9,8.0,Carrots,fresh vegetables,produce
6,2,40141,7,1,202279,3,5,9,8.0,Original Unflavored Gelatine Mix,doughs gelatins bake mixes,pantry
7,2,1819,8,1,202279,3,5,9,8.0,All Natural No Stir Creamy Almond Butter,spreads,pantry
8,2,43668,9,0,202279,3,5,9,8.0,Classic Blend Cole Slaw,packaged vegetables fruits,produce
9,3,33754,1,1,205970,16,5,17,12.0,Total 2% with Strawberry Lowfat Greek Strained...,yogurt,dairy eggs


## sampling


In [4]:
sample_order_ids = df['order_id'].drop_duplicates().sample(n=2000000, random_state=42)
df_sample = df[df['order_id'].isin(sample_order_ids)]

## الفلترة واستبعاد اشهر 10 منتجات


In [5]:
min_support = 0.00058
min_count = min_support * len(sample_order_ids)

item_counts = df_sample['product_id'].value_counts()
frequent_items = item_counts[item_counts >= min_count].index
top_selling_ids = item_counts.head(10).index
frequent_items_final = frequent_items[~frequent_items.isin(top_selling_ids)]

df_filtered_final = df_sample[df_sample['product_id'].isin(frequent_items_final)]
transactions_final = df_filtered_final.groupby('order_id')['product_id'].apply(list).tolist()

print(len(frequent_items_final), len(transactions_final))  # لازم يقرب من (1142, 1922950)

2978 1922950


In [6]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

te_final = TransactionEncoder()
te_ary_final = te_final.fit(transactions_final).transform(transactions_final, sparse=True)
basket_final = pd.DataFrame.sparse.from_spmatrix(te_ary_final, columns=te_final.columns_)
basket_final.columns = basket_final.columns.astype(str)

frequent_itemsets_final = fpgrowth(basket_final, min_support=min_support, use_colnames=True)

rules_final = association_rules(frequent_itemsets_final, metric="confidence", min_threshold=0.1)
rules_final = rules_final.sort_values('lift', ascending=False)

C:\Users\STH\AppData\Local\Temp\ipykernel_16800\3420726196.py:6: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_final = pd.DataFrame.sparse.from_spmatrix(te_ary_final, columns=te_final.columns_)


In [7]:
id_to_name = df.drop_duplicates('product_id').set_index('product_id')['product_name'].to_dict()
id_to_aisle = df.drop_duplicates('product_id').set_index('product_id')['aisle'].to_dict()
id_to_department = df.drop_duplicates('product_id').set_index('product_id')['department'].to_dict()

def map_names(frozenset_ids):
    return {id_to_name.get(int(i), i) for i in frozenset_ids}

def get_aisle_dept(id_set):
    aisles = {id_to_aisle.get(int(i)) for i in id_set}
    depts = {id_to_department.get(int(i)) for i in id_set}
    aisle = next(iter(aisles)) if len(aisles) == 1 else '/'.join(sorted(aisles))
    dept = next(iter(depts)) if len(depts) == 1 else '/'.join(sorted(depts))
    return aisle, dept

rules_final['antecedents_names'] = rules_final['antecedents'].apply(map_names)
rules_final['consequents_names'] = rules_final['consequents'].apply(map_names)

rules_final[['antecedent_aisle', 'antecedent_department']] = rules_final['antecedents'].apply(
    lambda s: pd.Series(get_aisle_dept(s))
)
rules_final[['consequent_aisle', 'consequent_department']] = rules_final['consequents'].apply(
    lambda s: pd.Series(get_aisle_dept(s))
)

rules_final['same_aisle'] = rules_final['antecedent_aisle'] == rules_final['consequent_aisle']
rules_final['same_department_diff_aisle'] = (
    (rules_final['antecedent_department'] == rules_final['consequent_department'])
    & (~rules_final['same_aisle'])
)

rules_cross_aisle = rules_final.sort_values('lift', ascending=False)
rules_cross_aisle[['antecedents_names', 'consequents_names', 'support', 'confidence', 'lift',
                    'same_aisle', 'same_department_diff_aisle']].head(20)

,antecedents_names,consequents_names,support,confidence,lift,same_aisle,same_department_diff_aisle
567,{Almond Milk Peach Yogurt},{Almond Milk Blueberry Yogurt},0.000747,0.482689,310.326633,True,False
566,{Almond Milk Blueberry Yogurt},{Almond Milk Peach Yogurt},0.000747,0.480107,310.326633,True,False
595,{Almond Milk Blueberry Yogurt},{Almond Milk Strawberry Yogurt},0.000880,0.566031,301.342783,True,False
594,{Almond Milk Strawberry Yogurt},{Almond Milk Blueberry Yogurt},0.000880,0.468715,301.342783,True,False
564,{Almond Milk Strawberry Yogurt},{Almond Milk Peach Yogurt},0.000818,0.435493,281.489372,True,False
565,{Almond Milk Peach Yogurt},{Almond Milk Strawberry Yogurt},0.000818,0.528739,281.489372,True,False
639,{Coconut Chia Bar},{Chocolate Peanut Butter},0.000626,0.409184,274.926536,True,False
640,{Chocolate Peanut Butter},{Coconut Chia Bar},0.000626,0.420335,274.926536,True,False
637,{Yotoddler Organic Pear Spinach Mango Yogurt},{Organic Whole Milk Strawberry Beet Berry Yogu...,0.000917,0.466526,226.484544,True,False
638,{Organic Whole Milk Strawberry Beet Berry Yogu...,{Yotoddler Organic Pear Spinach Mango Yogurt},0.000917,0.445090,226.484544,True,False


In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

rules_export = rules_cross_aisle.copy()  # ده التغيير الوحيد: rules_final → rules_cross_aisle
rules_export['antecedents_names'] = rules_export['antecedents_names'].apply(lambda s: ', '.join(sorted(s)))
rules_export['consequents_names'] = rules_export['consequents_names'].apply(lambda s: ', '.join(sorted(s)))
rules_export = rules_export.drop(columns=['antecedents', 'consequents'])  # frozensets مش متوافقة مع parquet

rules_export.to_parquet('../data/processed/association_rules.parquet', index=False)
print("✅ saved association_rules.parquet", rules_export.shape)

✅ saved association_rules.parquet (647, 20)
